<a href="https://colab.research.google.com/github/jarvis-2003/Dental_ai/blob/main/implant_site_analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implant Site Analysis Pipeline
**Google Drive → raw DICOM → NIfTI → site report**

Flow:
1. Mount Google Drive
2. Convert the **raw CBCT DICOM series** to NIfTI (one dedicated cell)
3. Load teeth + craniofacial **segmentation masks** (NIfTI already on Drive)
4. Run site analysis on the NIfTI volumes (not on DICOM slices)

The analysis pipeline stays NIfTI-based. DICOM is only an input conversion step.

> **Clinical note:** Outputs are planning aids. Do not treat as a treatment plan without clinician review. Voxel intensities are not assumed to be calibrated HU unless your export is known to be HU.


## 1. Install dependencies


In [4]:
%pip -q install nibabel numpy scipy scikit-image pandas matplotlib pydicom dicom2nifti SimpleITK


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 74.8 MB/s eta 0:00:00


## 2. Imports and label maps


In [1]:
import os, json, shutil
from pathlib import Path
from typing import Optional

import numpy as np
import nibabel as nib
import pandas as pd
from IPython.display import display

CRANIOFACIAL_LABELS = {
    1: "mandible", 2: "teeth_lower", 3: "skull", 4: "head",
    5: "sinus_maxillary", 6: "sinus_frontal", 7: "teeth_upper",
}

TEETH_LABELS = {
    1: "lower_jawbone", 2: "upper_jawbone",
    3: "left_inferior_alveolar_canal", 4: "right_inferior_alveolar_canal",
    5: "left_maxillary_sinus", 6: "right_maxillary_sinus",
    7: "pharynx", 8: "bridge", 9: "crown", 10: "implant",
    11: "upper_right_central_incisor_fdi11", 12: "upper_right_lateral_incisor_fdi12",
    13: "upper_right_canine_fdi13", 14: "upper_right_first_premolar_fdi14",
    15: "upper_right_second_premolar_fdi15", 16: "upper_right_first_molar_fdi16",
    17: "upper_right_second_molar_fdi17", 18: "upper_right_third_molar_fdi18",
    19: "upper_left_central_incisor_fdi21", 20: "upper_left_lateral_incisor_fdi22",
    21: "upper_left_canine_fdi23", 22: "upper_left_first_premolar_fdi24",
    23: "upper_left_second_premolar_fdi25", 24: "upper_left_first_molar_fdi26",
    25: "upper_left_second_molar_fdi27", 26: "upper_left_third_molar_fdi28",
    27: "lower_left_central_incisor_fdi31", 28: "lower_left_lateral_incisor_fdi32",
    29: "lower_left_canine_fdi33", 30: "lower_left_first_premolar_fdi34",
    31: "lower_left_second_premolar_fdi35", 32: "lower_left_first_molar_fdi36",
    33: "lower_left_second_molar_fdi37", 34: "lower_left_third_molar_fdi38",
    35: "lower_right_central_incisor_fdi41", 36: "lower_right_lateral_incisor_fdi42",
    37: "lower_right_canine_fdi43", 38: "lower_right_first_premolar_fdi44",
    39: "lower_right_second_premolar_fdi45", 40: "lower_right_first_molar_fdi46",
    41: "lower_right_second_molar_fdi47", 42: "lower_right_third_molar_fdi48",
    43: "left_mandibular_incisive_canal_fdi103",
    44: "right_mandibular_incisive_canal_fdi104",
    45: "lingual_canal",
}

FDI_TO_LABEL = {
    11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18,
    21: 19, 22: 20, 23: 21, 24: 22, 25: 23, 26: 24, 27: 25, 28: 26,
    31: 27, 32: 28, 33: 29, 34: 30, 35: 31, 36: 32, 37: 33, 38: 34,
    41: 35, 42: 36, 43: 37, 44: 38, 45: 39, 46: 40, 47: 41, 48: 42,
}

IMPLANT_DIAMETERS = [3.3, 3.5, 4.0, 4.5, 5.0, 5.5]
IMPLANT_LENGTHS = [8.0, 10.0, 11.5, 13.0, 14.0, 16.0]
print("FDI map size:", len(FDI_TO_LABEL))


FDI map size: 32


## 3. Mount Google Drive and set paths

Edit the three paths to match your Drive.

- `DICOM_DIR` — folder with the raw CBCT DICOM slices
- `TEETH_MASK_PATH` — teeth segmentation NIfTI
- `CRANIO_MASK_PATH` — craniofacial segmentation NIfTI

If a CBCT NIfTI already exists, set `EXISTING_CBCT_NIFTI` and conversion is skipped.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

# -------- USER CONFIG (edit these) --------
DRIVE_ROOT = Path("/content/drive/MyDrive/implant_case")

DICOM_DIR = DRIVE_ROOT / "dicom"
TEETH_MASK_PATH = DRIVE_ROOT / "teeth_segmentation.nii.gz"
CRANIO_MASK_PATH = DRIVE_ROOT / "craniofacial_structures_segmentation.nii.gz"
EXISTING_CBCT_NIFTI = DRIVE_ROOT / "cbct.nii.gz"  # optional; used if the file exists

OUT_DIR = DRIVE_ROOT / "implant_analysis_output"
NIFTI_DIR = OUT_DIR / "nifti"
NIFTI_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT.exists(), DRIVE_ROOT)
print("DICOM dir:", DICOM_DIR.exists(), DICOM_DIR)
print("Teeth mask:", TEETH_MASK_PATH.exists(), TEETH_MASK_PATH)
print("Cranio mask:", CRANIO_MASK_PATH.exists(), CRANIO_MASK_PATH)


Mounted at /content/drive
Drive root: True /content/drive/MyDrive/implant_case
DICOM dir: True /content/drive/MyDrive/implant_case/dicom
Teeth mask: True /content/drive/MyDrive/implant_case/teeth_segmentation.nii.gz
Cranio mask: True /content/drive/MyDrive/implant_case/craniofacial_structures_segmentation.nii.gz


## 4. Convert raw DICOM → NIfTI

This is the **only** DICOM step. After this, everything uses NIfTI.

Tries `dicom2nifti` first, then SimpleITK (often more reliable for CBCT).


In [5]:
import SimpleITK as sitk

CBCT_NIFTI_PATH = (
    Path(EXISTING_CBCT_NIFTI)
    if Path(EXISTING_CBCT_NIFTI).is_file()
    else (NIFTI_DIR / "cbct.nii.gz")
)


def convert_dicom_with_sitk(dicom_dir: Path, out_path: Path) -> Path:
    reader = sitk.ImageSeriesReader()
    search = dicom_dir
    series_ids = reader.GetGDCMSeriesIDs(str(search))
    if not series_ids:
        for sub in sorted(p for p in dicom_dir.rglob("*") if p.is_dir()):
            series_ids = reader.GetGDCMSeriesIDs(str(sub))
            if series_ids:
                search = sub
                break
    if not series_ids:
        raise RuntimeError(
            f"No DICOM series found under {dicom_dir}. "
            "Point DICOM_DIR at the folder that contains the .dcm slices."
        )
    print("Found series IDs:", series_ids)
    best_id, best_files = None, []
    for sid in series_ids:
        fnames = reader.GetGDCMSeriesFileNames(str(search), sid)
        if len(fnames) > len(best_files):
            best_id, best_files = sid, fnames
    print(f"Using series {best_id} with {len(best_files)} files")
    reader.SetFileNames(best_files)
    image = reader.Execute()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(image, str(out_path), useCompression=True)
    return out_path


def convert_dicom_with_dicom2nifti(dicom_dir: Path, out_dir: Path) -> Optional[Path]:
    import dicom2nifti
    tmp = out_dir / "_d2n_tmp"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)
    try:
        dicom2nifti.convert_directory(str(dicom_dir), str(tmp), compression=True, reorient=True)
        niis = list(tmp.glob("*.nii.gz")) + list(tmp.glob("*.nii"))
        if not niis:
            return None
        src = max(niis, key=lambda p: p.stat().st_size)
        dest = out_dir / "cbct.nii.gz"
        shutil.copy(src, dest)
        return dest
    except Exception as e:
        print("dicom2nifti failed:", type(e).__name__, e)
        return None


cached = NIFTI_DIR / "cbct.nii.gz"
if Path(EXISTING_CBCT_NIFTI).is_file():
    CBCT_NIFTI_PATH = Path(EXISTING_CBCT_NIFTI)
    print("Using existing CBCT NIfTI:", CBCT_NIFTI_PATH)
elif cached.is_file():
    CBCT_NIFTI_PATH = cached
    print("Using previously converted CBCT NIfTI:", CBCT_NIFTI_PATH)
else:
    if not DICOM_DIR.exists():
        raise FileNotFoundError(f"DICOM_DIR not found: {DICOM_DIR}")
    print("Converting DICOM to NIfTI from", DICOM_DIR)
    converted = convert_dicom_with_dicom2nifti(DICOM_DIR, NIFTI_DIR)
    if converted is None:
        print("Falling back to SimpleITK...")
        converted = convert_dicom_with_sitk(DICOM_DIR, NIFTI_DIR / "cbct.nii.gz")
    CBCT_NIFTI_PATH = converted
    print("Wrote", CBCT_NIFTI_PATH, "MB", round(CBCT_NIFTI_PATH.stat().st_size / 1e6, 2))

assert TEETH_MASK_PATH.is_file(), f"Teeth mask not found: {TEETH_MASK_PATH}"
assert CRANIO_MASK_PATH.is_file(), f"Cranio mask not found: {CRANIO_MASK_PATH}"
assert CBCT_NIFTI_PATH.is_file(), "CBCT NIfTI missing after conversion"
print("OK CBCT NIfTI:", CBCT_NIFTI_PATH)


Using previously converted CBCT NIfTI: /content/drive/MyDrive/implant_case/implant_analysis_output/nifti/cbct.nii.gz
OK CBCT NIfTI: /content/drive/MyDrive/implant_case/implant_analysis_output/nifti/cbct.nii.gz


In [7]:
import nibabel as nib
import numpy as np

p = "/content/drive/MyDrive/implant_case/implant_analysis_output/nifti/cbct.nii.gz"
img = nib.load(p)
data = np.asanyarray(img.dataobj)
print("shape:", data.shape)
print("spacing (mm):", img.header.get_zooms()[:3])
print("dtype:", data.dtype, "min/max:", float(data.min()), float(data.max()))

# A usable CBCT should look like (e.g.) 400–800 on each axis, all three dims >> 1
ok = all(s > 32 for s in data.shape[:3])
print("LOOKS_OK_3D:" if ok else "BAD_ALMOST_2D:", ok)

shape: (568, 568, 334, 1)
spacing (mm): (np.float32(0.3), np.float32(0.3), np.float32(0.3))
dtype: int16 min/max: -1000.0 3095.0
LOOKS_OK_3D: True


## 5. Load NIfTI, RAS-orient, align CBCT to the mask grid

If CBCT and the teeth mask differ in shape/affine, CBCT is resampled onto the **teeth-mask grid**. Mask labels are never resampled.


In [8]:
import numpy as np
import nibabel as nib


def load_nifti(path):
    return nib.load(str(path))


def ras(img):
    return nib.as_closest_canonical(img)


def as_3d(img):
    """(X,Y,Z,1) + 5x5 affine → 3D image with 4x4 affine."""
    data = np.asanyarray(img.dataobj)
    while data.ndim > 3 and data.shape[-1] == 1:
        data = np.squeeze(data, axis=-1)
    if data.ndim > 3:
        data = data[..., 0]
    if data.ndim != 3:
        raise ValueError(f"Expected 3D, got {data.shape}")
    affine = np.asarray(img.affine, dtype=float)[:4, :4]
    out = nib.Nifti1Image(data, affine)
    try:
        out.header.set_zooms(tuple(float(z) for z in img.header.get_zooms()[:3]))
    except Exception:
        pass
    return out, data


teeth_img, teeth_seg = as_3d(ras(load_nifti(TEETH_MASK_PATH)))
cranio_img, cranio_seg = as_3d(ras(load_nifti(CRANIO_MASK_PATH)))
cbct_img, cbct_vol = as_3d(ras(load_nifti(CBCT_NIFTI_PATH)))

teeth_seg = teeth_seg.astype(np.int32)
cranio_seg = cranio_seg.astype(np.int32)
cbct_vol = cbct_vol.astype(np.float32)
spacing = tuple(float(v) for v in teeth_img.header.get_zooms()[:3])


def affines_close(a, b, atol=1e-3):
    return np.allclose(np.asarray(a)[:4, :4], np.asarray(b)[:4, :4], atol=atol)


same_shape = cbct_vol.shape == teeth_seg.shape
same_affine = affines_close(cbct_img.affine, teeth_img.affine)

print("Teeth mask:", teeth_seg.shape, "spacing mm", spacing)
print("Cranio mask:", cranio_seg.shape)
print("CBCT:", cbct_vol.shape, "spacing", tuple(float(v) for v in cbct_img.header.get_zooms()[:3]))
print("Same shape:", same_shape, "| same affine:", same_affine)

if same_shape and not same_affine:
    # Same voxels — do NOT resample; just share the mask affine
    cbct_img = nib.Nifti1Image(cbct_vol, teeth_img.affine)
    print("Adopted teeth affine (no resample)")
elif not same_shape:
    from nibabel.processing import resample_from_to
    cbct_img = resample_from_to(cbct_img, (teeth_img.shape[:3], teeth_img.affine), order=1)
    cbct_vol = np.squeeze(np.asanyarray(cbct_img.dataobj).astype(np.float32))
    print("CBCT after resample:", cbct_vol.shape)

print("Unique teeth labels (first 40):", np.unique(teeth_seg)[:40])

Teeth mask: (568, 568, 334) spacing mm (0.30000001192092896, 0.30000001192092896, 0.30000001192092896)
Cranio mask: (568, 568, 334)
CBCT: (568, 568, 334) spacing (0.30000001192092896, 0.30000001192092896, 0.30000001192092896)
Same shape: True | same affine: True
Unique teeth labels (first 40): [ 0  1  2  3  4  5  6  7  9 11 12 13 14 15 17 18 19 20 21 22 23 24 25 26
 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42]


## 6. Geometry utilities


In [9]:
def get_label_mask(seg, label_id):
    return (seg == int(label_id)).astype(np.uint8)


def mask_centroid_vox(mask):
    coords = np.argwhere(mask > 0)
    if len(coords) == 0:
        return None
    return coords.mean(axis=0)


def mask_centroid_mm(mask, spacing):
    c = mask_centroid_vox(mask)
    if c is None:
        return None
    return c * np.asarray(spacing, dtype=float)


def nearest_catalog(value, catalog, prefer="down"):
    arr = np.asarray(catalog, dtype=float)
    if prefer == "down":
        cand = arr[arr <= value + 1e-6]
        return float(cand.max()) if len(cand) else float(arr.min())
    return float(arr[np.argmin(np.abs(arr - value))])


def clip_center_radius(shape, center, radius_vox):
    lo = np.maximum(np.floor(center - radius_vox), 0).astype(int)
    hi = np.minimum(np.ceil(center + radius_vox) + 1, shape).astype(int)
    return tuple(slice(int(a), int(b)) for a, b in zip(lo, hi))


def surface_extent_mm(mask, center, direction_vox, spacing):
    if mask.sum() == 0 or np.linalg.norm(direction_vox) < 1e-8:
        return None
    d = direction_vox / (np.linalg.norm(direction_vox) + 1e-12)
    sp = np.asarray(spacing, dtype=float)
    step_mm = 0.2
    step_vox = d * (step_mm / (np.linalg.norm(d * sp) + 1e-12))
    hits = []
    for sign in (-1.0, 1.0):
        p = center.astype(float).copy()
        last_inside = None
        for _ in range(400):
            p = p + sign * step_vox
            idx = np.round(p).astype(int)
            if np.any(idx < 0) or np.any(idx >= np.array(mask.shape)):
                break
            if mask[tuple(idx)] > 0:
                last_inside = p.copy()
            elif last_inside is not None:
                break
        if last_inside is not None:
            hits.append(last_inside)
    if len(hits) < 2:
        return None
    delta = (hits[1] - hits[0]) * sp
    return float(np.linalg.norm(delta))


## 7. Site analysis functions

Missing-site detection, neighbours, and gap use the **actual labels**.

Ridge width / height / safety use a **local ROI** around the interpolated neighbour centroid. If a measurement cannot be made, the field is `None` — no fake 7.0 / 11.5 / 4.8 values.


In [14]:
def detect_missing_sites(teeth_seg, spacing=(0.3, 0.3, 0.3)):
    present = set(int(x) for x in np.unique(teeth_seg))
    missing = []
    for fdi, lid in FDI_TO_LABEL.items():
        if lid not in present:
            missing.append({
                "fdi": fdi,
                "site_id": f"Site_{fdi}",
                "label_id": lid,
                "status": "missing",
            })
    missing.sort(key=lambda x: x["fdi"])
    print(f"Detected {len(missing)} missing sites:", [m["fdi"] for m in missing])
    return missing


def classify_jaw_and_site(site, teeth_seg, cranio_seg):
    fdi = site["fdi"]
    jaw = "maxilla" if 11 <= fdi <= 28 else "mandible"
    jawbone_label = 2 if jaw == "maxilla" else 1
    side = "right" if fdi in list(range(11, 19)) + list(range(41, 49)) else "left"
    site.update(
        jaw=jaw,
        side=side,
        jawbone_label=jawbone_label,
        is_terminal=fdi in (18, 28, 38, 48),
    )
    return site


def get_adjacent_fdi(fdi):
    order = {
        "upper_right": [18, 17, 16, 15, 14, 13, 12, 11],
        "upper_left": [21, 22, 23, 24, 25, 26, 27, 28],
        "lower_left": [38, 37, 36, 35, 34, 33, 32, 31],
        "lower_right": [41, 42, 43, 44, 45, 46, 47, 48],
    }
    for teeth in order.values():
        if fdi in teeth:
            idx = teeth.index(fdi)
            return {
                "mesial": teeth[idx - 1] if idx > 0 else None,
                "distal": teeth[idx + 1] if idx < len(teeth) - 1 else None,
            }
    return {"mesial": None, "distal": None}


def analyze_adjacent_teeth(site, teeth_seg):
    adj = get_adjacent_fdi(site["fdi"])
    present = set(int(x) for x in np.unique(teeth_seg))
    result = {}
    for direction in ("mesial", "distal"):
        fdi = adj[direction]
        if fdi is None:
            result[direction] = {"fdi": None, "status": "end_of_arch"}
        else:
            lid = FDI_TO_LABEL[fdi]
            result[direction] = {
                "fdi": fdi,
                "status": "present" if lid in present else "missing",
                "label_id": lid,
            }
    site["adjacent"] = result
    site["is_free_end"] = any(result[d]["status"] == "end_of_arch" for d in result)
    site["is_edentulous_span"] = all(result[d]["status"] == "missing" for d in result)
    return site


def site_center_and_axes(site, teeth_seg, spacing):
    spacing = np.asarray(spacing, dtype=float)
    adj = site.get("adjacent", {})
    pts = []
    for key in ("mesial", "distal"):
        info = adj.get(key) or {}
        if info.get("status") == "present":
            c = mask_centroid_vox(get_label_mask(teeth_seg, info["label_id"]))
            if c is not None:
                pts.append(c)

    bone = get_label_mask(teeth_seg, site["jawbone_label"])
    if len(pts) == 2:
        center = 0.5 * (pts[0] + pts[1])
        md = pts[1] - pts[0]
    elif len(pts) == 1:
        center = pts[0]
        md = None
    else:
        center = mask_centroid_vox(bone)
        md = None

    if center is None:
        return None

    aff = teeth_img.affine
    axis_world = aff[:3, :3]
    si_axis = int(np.argmax(np.abs(axis_world[2, :])))
    is_dir = np.zeros(3)
    is_dir[si_axis] = 1.0
    if axis_world[2, si_axis] < 0:
        is_dir *= -1

    if md is None or np.linalg.norm(md) < 1e-6:
        md = np.zeros(3)
        md[(si_axis + 1) % 3] = 1.0
    md = md - np.dot(md, is_dir) * is_dir
    if np.linalg.norm(md) < 1e-6:
        md = np.zeros(3)
        md[(si_axis + 1) % 3] = 1.0
    md = md / (np.linalg.norm(md) + 1e-12)
    is_u = is_dir / (np.linalg.norm(is_dir) + 1e-12)
    bl = np.cross(is_u, md)
    bl = bl / (np.linalg.norm(bl) + 1e-12)

    idx = tuple(np.clip(np.round(center).astype(int), 0, np.array(bone.shape) - 1))
    if bone.sum() and bone[idx] == 0:
        coords = np.argwhere(bone > 0)
        d2 = ((coords - center) ** 2).sum(axis=1)
        center = coords[int(np.argmin(d2))].astype(float)

    site["geometry"] = {
        "center_vox": center.tolist(),
        "md_axis_vox": md.tolist(),
        "is_axis_vox": is_u.tolist(),
        "bl_axis_vox": bl.tolist(),
        "si_array_axis": si_axis,
    }
    return center, md, is_u, bl, bone


def calculate_ridge_width(site, teeth_seg, spacing, heights_mm=(0, 3, 5)):
    got = site_center_and_axes(site, teeth_seg, spacing)
    if got is None:
        site["ridge_width"] = {
            "measurement_status": "no_center",
            "bone_width_mm": None,
            "horizontal_deficiency_mm": None,
            "classification": None,
        }
        return site
    center, md, is_u, bl, bone = got
    sp = np.asarray(spacing, dtype=float)
    widths = {}
    for h in heights_mm:
        probe = center - is_u * (h / (np.linalg.norm(is_u * sp) + 1e-12))
        sl = clip_center_radius(bone.shape, probe, radius_vox=np.array([8.0 / max(s, 1e-6) for s in sp]))
        local = np.zeros_like(bone)
        local[sl] = bone[sl]
        w = surface_extent_mm(local, probe, bl, spacing)
        widths[f"width_at_{h}mm"] = None if w is None else round(float(w), 2)

    measured = [v for v in widths.values() if v is not None]
    narrowest = min(measured) if measured else None
    ref = 6.0
    hdef = None if narrowest is None else round(max(0.0, ref - narrowest), 2)
    site["ridge_width"] = {
        **widths,
        "bone_width_mm": narrowest,
        "narrowest_mm": narrowest,
        "horizontal_deficiency_mm": hdef,
        "classification": None if hdef is None else ("adequate" if hdef < 1 else "deficient"),
        "measurement_status": "measured_from_site_roi" if narrowest is not None else "failed_local_width",
    }
    return site


def last_hit(mask, start, direction, spacing, max_mm=40):
    sp = np.asarray(spacing, dtype=float)
    d = direction / (np.linalg.norm(direction) + 1e-12)
    step_mm = 0.2
    step = d * (step_mm / (np.linalg.norm(d * sp) + 1e-12))
    p = start.astype(float).copy()
    last = None
    for _ in range(int(max_mm / step_mm)):
        p = p + step
        idx = np.round(p).astype(int)
        if np.any(idx < 0) or np.any(idx >= np.array(mask.shape)):
            break
        if mask[tuple(idx)] > 0:
            last = p.copy()
    return last


def first_hit(mask, start, direction, spacing, max_mm=40):
    sp = np.asarray(spacing, dtype=float)
    d = direction / (np.linalg.norm(direction) + 1e-12)
    step_mm = 0.2
    step = d * (step_mm / (np.linalg.norm(d * sp) + 1e-12))
    p = start.astype(float).copy()
    for _ in range(int(max_mm / step_mm)):
        p = p + step
        idx = np.round(p).astype(int)
        if np.any(idx < 0) or np.any(idx >= np.array(mask.shape)):
            return None
        if mask[tuple(idx)] > 0:
            return p.copy()
    return None


def calculate_ridge_height(site, teeth_seg, cranio_seg, spacing):
    got = site.get("geometry")
    if not got:
        got_full = site_center_and_axes(site, teeth_seg, spacing)
        if got_full is None:
            site["ridge_height"] = {
                "measurement_status": "no_center",
                "bone_height_mm": None,
                "vertical_deficiency_mm": None,
                "classification": None,
            }
            return site
        center, md, is_u, bl, bone = got_full
    else:
        center = np.asarray(got["center_vox"], dtype=float)
        is_u = np.asarray(got["is_axis_vox"], dtype=float)
        bone = get_label_mask(teeth_seg, site["jawbone_label"])

    sp = np.asarray(spacing, dtype=float)
    if site["jaw"] == "mandible":
        landmark = get_label_mask(teeth_seg, 3 if site["side"] == "left" else 4)
        landmark_name = "ian_canal"
    else:
        landmark = get_label_mask(teeth_seg, 5 if site["side"] == "left" else 6)
        if landmark.sum() == 0:
            landmark = (cranio_seg == 5).astype(np.uint8)
        landmark_name = "sinus_floor"
    ref = 10.0

    crest = last_hit(bone, center, is_u, spacing)
    start = crest if crest is not None else center
    land_pt = first_hit(landmark, start, -is_u, spacing) if landmark.sum() else None
    height = None
    if crest is not None and land_pt is not None:
        height = float(np.linalg.norm((crest - land_pt) * sp))
    elif crest is not None:
        bone_end = last_hit(bone, start, -is_u, spacing)
        if bone_end is not None:
            height = float(np.linalg.norm((crest - bone_end) * sp))

    vdef = None if height is None else round(max(0.0, ref - height), 2)
    site["ridge_height"] = {
        "bone_height_mm": None if height is None else round(height, 2),
        "usable_height_mm": None if height is None else round(height, 2),
        "landmark": landmark_name,
        "vertical_deficiency_mm": vdef,
        "classification": None if vdef is None else ("adequate" if vdef < 1 else "deficient"),
        "measurement_status": "measured_crest_to_landmark" if height is not None else "failed_height",
    }
    return site


def calculate_edentulous_gap(site, teeth_seg, spacing):
    adj = site.get("adjacent", {})
    mesial, distal = adj.get("mesial", {}), adj.get("distal", {})
    gap = {
        "mesial_fdi": mesial.get("fdi"),
        "distal_fdi": distal.get("fdi"),
        "gap_md_mm": None,
        "gap_area_mm2": None,
        "gap_volume_mm3": None,
        "status": "ok",
    }
    if mesial.get("status") != "present" or distal.get("status") != "present":
        gap["status"] = "cannot_measure_full_gap (free-end or span)"
        site["edentulous_gap"] = gap
        return site

    m = get_label_mask(teeth_seg, mesial["label_id"])
    d = get_label_mask(teeth_seg, distal["label_id"])
    cm = mask_centroid_mm(m, spacing)
    cd = mask_centroid_mm(d, spacing)
    if cm is None or cd is None:
        gap["status"] = "neighbour_centroid_missing"
        site["edentulous_gap"] = gap
        return site

    sp = np.asarray(spacing, dtype=float)
    c0 = mask_centroid_vox(m)
    c1 = mask_centroid_vox(d)
    md = (c1 - c0)
    md = md / (np.linalg.norm(md) + 1e-12)

    p_m = last_hit(m, c0, md, spacing, max_mm=20)
    p_d = last_hit(d, c1, -md, spacing, max_mm=20)
    if p_m is None or p_d is None:
        gap_md = float(np.linalg.norm(cm - cd))
        status = "centroid_distance_fallback"
    else:
        gap_md = float(np.linalg.norm((p_d - p_m) * sp))
        status = "surface_to_surface_along_md"

    width = site.get("ridge_width", {}).get("narrowest_mm")
    height = site.get("ridge_height", {}).get("usable_height_mm")
    area = None if width is None else gap_md * float(width)
    vol = None if (area is None or height is None) else area * float(height)
    gap.update(
        centroid_distance_mm=round(float(np.linalg.norm(cm - cd)), 2),
        gap_md_mm=round(gap_md, 2),
        gap_area_mm2=None if area is None else round(area, 2),
        gap_volume_mm3=None if vol is None else round(vol, 1),
        status=status,
    )
    site["edentulous_gap"] = gap
    return site


def calculate_bone_density(site, cbct_volume, teeth_seg, spacing):
    if cbct_volume is None:
        site["bone_density"] = {"status": "cbct_volume_not_provided", "density_class": None}
        return site
    if cbct_volume.shape != teeth_seg.shape:
        site["bone_density"] = {"status": "grid_mismatch", "density_class": None}
        return site
    bone = get_label_mask(teeth_seg, site["jawbone_label"])
    geo = site.get("geometry") or {}
    center = np.asarray(geo.get("center_vox") or mask_centroid_vox(bone) or [0, 0, 0], dtype=float)
    sl = clip_center_radius(bone.shape, center, radius_vox=np.array([8.0 / max(s, 1e-6) for s in spacing]))
    local = np.zeros_like(bone)
    local[sl] = bone[sl]
    vals = cbct_volume[local > 0]
    if vals.size == 0:
        site["bone_density"] = {"status": "no_bone_voxels_in_site_roi", "density_class": None}
        return site
    mean = float(vals.mean())
    std = float(vals.std())
    dens = "D1" if mean > 1250 else "D2" if mean > 850 else "D3" if mean > 350 else "D4"
    site["bone_density"] = {
        "mean_intensity": round(mean, 1),
        "std_intensity": round(std, 1),
        "density_class": dens,
        "volume_voxels": int(local.sum()),
        "status": "raw intensity at site ROI; treat as HU only if the NIfTI is HU-calibrated",
    }
    return site


def calculate_safety_landmarks(site, teeth_seg, spacing):
    got = site.get("geometry")
    if not got:
        site_center_and_axes(site, teeth_seg, spacing)
        got = site.get("geometry")
    if not got:
        site["safety"] = {"measurement_status": "no_center"}
        return site

    center = np.asarray(got["center_vox"], dtype=float)
    sp = np.asarray(spacing, dtype=float)

    if site["jaw"] == "mandible":
        landmark = get_label_mask(teeth_seg, 3 if site["side"] == "left" else 4)
        name = "ian_canal"
        thresh = 2.0
    else:
        landmark = get_label_mask(teeth_seg, 5 if site["side"] == "left" else 6)
        if landmark.sum() == 0:
            landmark = (cranio_seg == 5).astype(np.uint8)
        name = "maxillary_sinus"
        thresh = 1.0

    if landmark.sum() == 0:
        site["safety"] = {
            "type": name,
            "distance_mm": None,
            "safety_margin_ok": None,
            "measurement_status": "landmark_label_absent",
        }
        return site

    coords = np.argwhere(landmark > 0).astype(float)
    dist = float(np.min(np.linalg.norm((coords - center) * sp, axis=1)))
    site["safety"] = {
        "type": name,
        "distance_mm": round(dist, 2),
        "safety_margin_ok": dist >= thresh,
        "measurement_status": "min_distance_center_to_landmark",
    }
    return site


def calculate_ridge_resorption(site):
    hw, hh = site.get("ridge_width", {}), site.get("ridge_height", {})
    h, v = hw.get("horizontal_deficiency_mm"), hh.get("vertical_deficiency_mm")
    if h is None and v is None:
        grade = None
    else:
        h = h or 0
        v = v or 0
        grade = "minimal / none" if h < 1 and v < 1 else "moderate" if h < 3 and v < 3 else "severe"
    site["ridge_resorption"] = {
        "horizontal_deficiency_mm": hw.get("horizontal_deficiency_mm"),
        "vertical_deficiency_mm": hh.get("vertical_deficiency_mm"),
        "grade": grade,
        "note": "Vs typical implant envelope, not vs a pre-extraction scan.",
    }
    return site


def calculate_augmented_volume(site, teeth_seg, spacing, preop_bone_mask=None):
    hw, hh, gap = site.get("ridge_width", {}), site.get("ridge_height", {}), site.get("edentulous_gap", {})
    hd = float(hw.get("horizontal_deficiency_mm") or 0)
    vd = float(hh.get("vertical_deficiency_mm") or 0)
    md = float(gap.get("gap_md_mm") or 7.0)
    height = float(hh.get("usable_height_mm") or 10.0)
    width = float(hw.get("narrowest_mm") or 6.0)
    vol = max(0.0, hd * md * height + vd * md * width)
    jaw = site.get("jaw")
    if vol < 1 and hd < 1 and vd < 1:
        aug = "none"
    elif jaw == "maxilla" and vd >= 5:
        aug = "sinus_lift"
    elif jaw == "maxilla" and vd >= 3:
        aug = "sinus_lift_or_GBR"
    elif hd >= 4 or vd >= 5:
        aug = "block_graft"
    elif hd >= 1 or vd >= 1:
        aug = "GBR"
    else:
        aug = "none"
    result = {
        "needed_volume_mm3": round(vol, 1),
        "grafted_volume_mm3": round(vol, 1),
        "augmentation_type": aug,
        "status": "estimated_from_deficiency",
    }
    if preop_bone_mask is not None:
        bone = get_label_mask(teeth_seg, site["jawbone_label"])
        current = float(np.sum(bone)) * float(np.prod(spacing))
        preop = float(np.sum(preop_bone_mask)) * float(np.prod(spacing))
        result.update(
            current_bone_mm3=round(current, 1),
            preop_bone_mm3=round(preop, 1),
            grafted_volume_mm3=round(current - preop, 1),
            status="preop_minus_current",
        )
    site["augmented_volume"] = result
    return site


def recommend_implant_size(site):
    width = site.get("ridge_width", {}).get("narrowest_mm")
    height = site.get("ridge_height", {}).get("usable_height_mm")
    sd = (site.get("safety") or {}).get("distance_mm")
    if width is None or height is None:
        site["implant_size"] = {
            "implant_diameter_mm": None,
            "implant_length_mm": None,
            "note": "Skipped — missing measured width/height.",
        }
        return site
    sd = float(sd) if sd is not None else float(height)
    maxd = max(0.0, float(width) - 3.0)
    maxl = max(0.0, min(float(height), sd) - 1.5)
    site["implant_size"] = {
        "implant_diameter_mm": nearest_catalog(maxd, IMPLANT_DIAMETERS),
        "implant_length_mm": nearest_catalog(maxl, IMPLANT_LENGTHS),
        "max_diameter_from_ridge_mm": round(maxd, 2),
        "max_length_from_height_mm": round(maxl, 2),
        "note": "Catalog snap-down from measured envelope. Clinician must confirm.",
    }
    return site


def estimate_healing_timeline(site):
    dens = (site.get("bone_density") or {}).get("density_class")
    aug = (site.get("augmented_volume") or {}).get("augmentation_type", "none")
    base = {"D1": 3, "D2": 4, "D3": 6, "D4": 8}.get(dens, 6)
    extra = 4 if aug == "GBR" else 6 if aug in ("block_graft", "sinus_lift", "sinus_lift_or_GBR") else 0
    months = int(min(12, max(3, base + extra)))
    site["healing_timeline"] = {
        "timeline_months": months,
        "density_base_months": base,
        "graft_extra_months": extra,
        "augmentation_type": aug,
        "note": "Illustrative estimate only.",
    }
    return site


def assess_risk_level(site):
    flags, score = [], 0
    hw, hh = site.get("ridge_width", {}), site.get("ridge_height", {})
    dens = (site.get("bone_density") or {}).get("density_class")
    safety_ok = (site.get("safety") or {}).get("safety_margin_ok", True)
    if hw.get("classification") == "deficient":
        flags.append("horizontal_bone_deficiency")
        score += 1 if (hw.get("horizontal_deficiency_mm") or 0) < 3 else 2
    if hh.get("classification") == "deficient":
        flags.append("vertical_bone_deficiency")
        score += 1 if (hh.get("vertical_deficiency_mm") or 0) < 3 else 2
    if dens == "D4":
        flags.append("low_bone_density_D4")
        score += 2
    elif dens == "D3":
        score += 1
    if safety_ok is False:
        flags.append("insufficient_safety_margin")
        score += 3
    if site.get("ridge_resorption", {}).get("grade") == "severe":
        flags.append("severe_ridge_resorption")
        score += 2
    if site.get("augmented_volume", {}).get("augmentation_type") in ("block_graft", "sinus_lift"):
        score += 1
    if safety_ok is False or score >= 4:
        level = "High"
    elif score >= 2:
        level = "Medium"
    else:
        level = "Low"
    site["risk"] = {"risk_level": level, "score": score, "flags": flags}
    site["implant_readiness_flags"] = flags
    return site


## 8. Compile reports and run


In [13]:
def compile_site_report(site):
    flags = site.get("implant_readiness_flags") or (site.get("risk") or {}).get("flags") or []
    return {
        "site_id": site.get("site_id", f"Site_{site['fdi']}"),
        "site_fdi": site["fdi"],
        "jaw": site["jaw"],
        "side": site["side"],
        "adjacent_teeth": site.get("adjacent"),
        "is_free_end": site.get("is_free_end"),
        "is_edentulous_span": site.get("is_edentulous_span"),
        "edentulous_gap": site.get("edentulous_gap"),
        "ridge_width": site.get("ridge_width"),
        "ridge_height": site.get("ridge_height"),
        "bone_density": site.get("bone_density"),
        "safety_landmarks": site.get("safety"),
        "ridge_resorption": site.get("ridge_resorption"),
        "augmented_volume": site.get("augmented_volume"),
        "implant_size": site.get("implant_size"),
        "healing_timeline": site.get("healing_timeline"),
        "risk": site.get("risk"),
        "implant_readiness_flags": flags,
        "overall_status": "review_required" if flags else "candidate",
        "table_row": {
            "Site ID": site.get("site_id", f"Site_{site['fdi']}"),
            "Missing Tooth FDI": site["fdi"],
            "Bone Height (mm)": (site.get("ridge_height") or {}).get("bone_height_mm"),
            "Bone Width (mm)": (site.get("ridge_width") or {}).get("bone_width_mm"),
            "Vertical Deficiency (mm)": (site.get("ridge_height") or {}).get("vertical_deficiency_mm"),
            "Horizontal Deficiency (mm)": (site.get("ridge_width") or {}).get("horizontal_deficiency_mm"),
            "Aug Volume (mm³)": (site.get("augmented_volume") or {}).get("grafted_volume_mm3"),
            "Timeline (months)": (site.get("healing_timeline") or {}).get("timeline_months"),
            "Implant Length (mm)": (site.get("implant_size") or {}).get("implant_length_mm"),
            "Implant Diameter (mm)": (site.get("implant_size") or {}).get("implant_diameter_mm"),
            "Augmentation Type": (site.get("augmented_volume") or {}).get("augmentation_type"),
            "Risk Level": (site.get("risk") or {}).get("risk_level"),
        },
    }


def run_full_pipeline(teeth_seg, cranio_seg, cbct_volume=None, spacing=(0.3, 0.3, 0.3), preop_bone_mask=None):
    missing = detect_missing_sites(teeth_seg, spacing)
    reports = []
    for site in missing:
        print(f"\n=== FDI {site['fdi']} ===")
        site = classify_jaw_and_site(site, teeth_seg, cranio_seg)
        site = analyze_adjacent_teeth(site, teeth_seg)
        site = calculate_ridge_width(site, teeth_seg, spacing)
        site = calculate_ridge_height(site, teeth_seg, cranio_seg, spacing)
        site = calculate_edentulous_gap(site, teeth_seg, spacing)
        site = calculate_bone_density(site, cbct_volume, teeth_seg, spacing)
        site = calculate_safety_landmarks(site, teeth_seg, spacing)
        site = calculate_ridge_resorption(site)
        site = calculate_augmented_volume(site, teeth_seg, spacing, preop_bone_mask)
        site = recommend_implant_size(site)
        site = estimate_healing_timeline(site)
        site = assess_risk_level(site)
        reports.append(compile_site_report(site))
    return reports


reports = run_full_pipeline(teeth_seg, cranio_seg, cbct_vol, spacing)
table = pd.DataFrame([r["table_row"] for r in reports])
display(table)
print("Reports:", len(reports))


Detected 1 missing sites: [16]

=== FDI 16 ===
m is [[[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 ...

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]] and n is [[[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 

,Site ID,Missing Tooth FDI,Bone Height (mm),Bone Width (mm),Vertical Deficiency (mm),Horizontal Deficiency (mm),Aug Volume (mm³),Timeline (months),Implant Length (mm),Implant Diameter (mm),Augmentation Type,Risk Level
0,Site_16,16,7.2,4.2,2.8,1.8,246.2,10,8.0,3.3,GBR,Medium


Reports: 1


## 9. Save CSV + JSON back to Google Drive


In [11]:
csv_path = OUT_DIR / "implant_site_summary.csv"
json_path = OUT_DIR / "implant_site_reports.json"
table.to_csv(csv_path, index=False)


def _jsonable(x):
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    return str(x)


with open(json_path, "w") as f:
    json.dump(reports, f, indent=2, default=_jsonable)

print("Saved to Drive:")
print(" ", csv_path)
print(" ", json_path)
print(" ", "CBCT NIfTI:", CBCT_NIFTI_PATH)


Saved to Drive:
  /content/drive/MyDrive/implant_case/implant_analysis_output/implant_site_summary.csv
  /content/drive/MyDrive/implant_case/implant_analysis_output/implant_site_reports.json
  CBCT NIfTI: /content/drive/MyDrive/implant_case/implant_analysis_output/nifti/cbct.nii.gz


## 10. What changed vs the previous notebook

| Issue | Change |
|-------|--------|
| `files.upload()` | Removed. Paths come from **Google Drive** after `drive.mount` |
| Raw DICOM treated as NIfTI | Dedicated **DICOM → NIfTI** cell (`dicom2nifti`, SimpleITK fallback). Analysis stays NIfTI |
| Hardcoded ridge 7.0 / 11.5 and safety 4.8 / 7.2 | Site-ROI width, crest-to-canal/sinus height, min-distance safety. `None` if unmeasurable |
| CBCT vs mask grid mismatch | CBCT is **resampled onto the teeth-mask grid**; labels are never interpolated |
| Implicit FDI map (`fdi-2`) | Explicit `FDI_TO_LABEL` matching your label table |
| Gap = centroid minus 8 mm | Surface-to-surface along the mesial–distal axis |
| Density on the whole jaw | Density sampled in the **site ROI** |

**Expected Drive layout**

```
MyDrive/implant_case/
  dicom/                         <- raw CBCT slices
  teeth_segmentation.nii.gz
  craniofacial_mask.nii.gz
  implant_analysis_output/       <- created automatically
    nifti/cbct.nii.gz
    implant_site_summary.csv
    implant_site_reports.json
```
